# GSB 5544 — Pandas Fundamentals: Asking Questions of a DataFrame  
**SOLUTION VERSION**

Every section of this notebook starts with a **question**. The pandas tool we learn is simply the way to answer it. When you study, try to remember the *question* — the code will follow.

**Data set:** `coffee_purchases.csv` — every coffee purchase pulled from a bank statement (2018–2025). Each **row** is one purchase; each **column** is one thing we know about that purchase.

| Column | What it means |
|---|---|
| `date` | day of purchase, stored as *number of days since Jan 1, 1970* (we'll fix this!) |
| `description` | raw text from the bank statement |
| `amount` | dollars (negative = money leaving the account) |
| `account` | which account paid |
| `month`, `year`, `day_of_week` | when the purchase happened |

## 0. Setup — *How do I get a CSV file into Python?*

`pandas` is the library that gives Python a table (a `DataFrame`). `pd.read_csv()` turns a CSV file into one.

> **Instructor:** paste the raw URL of `coffee_purchases.csv` (GitHub raw link or Google Drive share) inside the quotes below.

In [1]:
import pandas as pd

coffee = pd.read_csv("https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/coffee_purchases.csv")   # <-- put the CSV URL / path here
coffee.head()

,date,description,amount,account,month,year,day_of_week
0,17826,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday
1,17833,SQ *LUCY'S COFFEE C 10/27 PURCHASE SAN LUIS OB...,-7.43,Eman,October,2018,Monday
2,18050,SQ *LUCY'S COFFEE CO 06/02 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday
3,18057,SQ *LUCY'S COFFEE CO 06/08 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday
4,18136,SQ *SCOUT COFFEE 08/27 PURCHASE SAN LUIS OBIS CA,-9.21,Kelley,August,2019,Wednesday


*(Alternative if you have the file on your computer: run the cell below, pick the file, then use `pd.read_csv("coffee_purchases.csv")`.)*

In [2]:
# from google.colab import files
# files.upload()
# coffee = pd.read_csv("coffee_purchases.csv")

---
## 1. Looking at the data — *How big is this data set, and what is in it?*

Before analyzing anything, answer three questions:

1. **How many rows and columns?** → `.shape`
2. **What kind of object am I holding?** → `type()`
3. **What kind of values are in each column?** → `.dtypes` (and `.info()`)

In [3]:
# Q: How many purchases (rows) and how many variables (columns)?
coffee.shape          # (rows, columns)

(821, 7)

In [4]:
# .shape is a tuple, so you can grab each piece
print("rows:", coffee.shape[0])
print("cols:", coffee.shape[1])
print("rows (another way):", len(coffee))

rows: 821
cols: 7
rows (another way): 821


In [5]:
# Q: What kind of Python object is `coffee`?
type(coffee)

<class 'pandas.core.frame.DataFrame'>

In [6]:
# Q: What kind of values live in each column?
coffee.dtypes

date             int64
description     object
amount         float64
account         object
month           object
year             int64
day_of_week     object
dtype: object

Pandas' storage types are not the same as the *statistical* variable types we care about:

| pandas `dtype` | means |
|---|---|
| `int64` | whole numbers |
| `float64` | decimal numbers |
| `object` | usually text (strings) |
| `category` | a categorical variable with a fixed set of levels |
| `datetime64` | a real date/time |

Notice `date` is `int64`. Pandas *thinks* it is a number. **We know better** — that's the next section.

In [7]:
# .info() bundles shape, dtypes, and count of non-missing values into one report
coffee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 821 entries, 0 to 820
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         821 non-null    int64  
 1   description  821 non-null    object 
 2   amount       821 non-null    float64
 3   account      821 non-null    object 
 4   month        821 non-null    object 
 5   year         821 non-null    int64  
 6   day_of_week  821 non-null    object 
dtypes: float64(1), int64(2), object(4)
memory usage: 45.0+ KB


In [8]:
# Other quick looks
coffee.head(3)        # first 3 rows

,date,description,amount,account,month,year,day_of_week
0,17826,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday
1,17833,SQ *LUCY'S COFFEE C 10/27 PURCHASE SAN LUIS OB...,-7.43,Eman,October,2018,Monday
2,18050,SQ *LUCY'S COFFEE CO 06/02 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday


In [9]:
coffee.tail(3)        # last 3 rows

,date,description,amount,account,month,year,day_of_week
818,20312,SQ *SCOUT COFFEE 08/11 MOBILE PURCHASE San Lui...,-55.00,Eman,August,2025,Tuesday
819,20305,SQ *SCOUT COFFEE San Luis Obis CA,-4.95,Credit Card,August,2025,Tuesday
820,20305,SQ *SCOUT COFFEE San Luis Obis CA,-6.50,Credit Card,August,2025,Tuesday


In [10]:
coffee.columns        # just the column names

Index(['date', 'description', 'amount', 'account', 'month', 'year',
       'day_of_week'],
      dtype='object')

### ✅ Check yourself
1. How many purchases are in the data set?
2. Which columns does pandas store as text (`object`)?
3. Which column has a dtype that is *misleading* about what the variable really is?

---
## 2. Variable types — *What kind of variable is each column, really?*

The dtype tells you how the computer stores a column. **You** decide what the variable *means*:

| Type | Definition | Example question it answers |
|---|---|---|
| **Quantitative** | Numbers where arithmetic makes sense (adding, averaging) | "What's the average amount spent?" |
| **Categorical** | Values are labels/groups (levels). Even if stored as numbers! | "Which account is used most?" |
| **Other: time & date** | Values are points in time; order and gaps matter | "How much did I spend per month in 2023?" |

Ask of each column: *does the average of this column mean anything?* If yes → quantitative. If the values are group names → categorical. If it's a calendar/clock value → date/time.

In [11]:
# Q: What type is each variable by DEFINITION (not by dtype)?
coffee.dtypes

date             int64
description     object
amount         float64
account         object
month           object
year             int64
day_of_week     object
dtype: object

| Column | dtype | Variable type by definition |
|---|---|---|
| `date` | int64 | **date/time** (stored as days since 1970-01-01) |
| `description` | object | text / identifier (categorical-ish, but ~every value is unique) |
| `amount` | float64 | **quantitative** |
| `account` | object | **categorical** |
| `month` | object | **categorical** (ordered!) |
| `year` | int64 | **categorical** (ordered) — the *average year* isn't meaningful |
| `day_of_week` | object | **categorical** (ordered) |

Now let's make the DataFrame **state** these types itself.

In [12]:
# Make `date` a real date. The integers are days since 1970-01-01 ("unit='D'").
coffee["date"] = pd.to_datetime(coffee["date"], unit="D")
coffee["date"].head()

0   2018-10-22
1   2018-10-29
2   2019-06-03
3   2019-06-10
4   2019-08-28
Name: date, dtype: datetime64[ns]

In [13]:
# Make the categorical variables actually categorical
coffee["account"] = coffee["account"].astype("category")

# For ordered categories, give pandas the order of the levels
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
coffee["month"] = pd.Categorical(coffee["month"], categories=month_order, ordered=True)

day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
coffee["day_of_week"] = pd.Categorical(coffee["day_of_week"], categories=day_order, ordered=True)

# Now the DataFrame *states* the variable types
coffee.dtypes

date           datetime64[ns]
description            object
amount                float64
account              category
month                category
year                    int64
day_of_week          category
dtype: object

In [14]:
# Q: Why does declaring a categorical matter? Because pandas can now answer "which levels exist, in order?"
coffee["day_of_week"].cat.categories

Index(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday',
       'Sunday'],
      dtype='object')

In [15]:
# ...and value_counts() / sorting respect the level order
coffee["day_of_week"].value_counts().sort_index()

day_of_week
Monday       373
Tuesday      129
Wednesday     93
Thursday     100
Friday       100
Saturday      15
Sunday        11
Name: count, dtype: int64

In [16]:
# Q: And why make `date` a real date? Because now we can pull out pieces of it.
coffee["date"].dt.year.head()

0    2018
1    2018
2    2019
3    2019
4    2019
Name: date, dtype: int32

In [17]:
# Q: Does the average of a quantitative variable make sense? (yes)
coffee["amount"].mean()

np.float64(-6.059037758830693)

In [18]:
# Q: Does the average of `year` make sense? Pandas will compute it... but the answer is meaningless.
# This is why *you* decide the variable type, not the computer.
coffee["year"].mean()

np.float64(2022.3848964677222)

### ✅ Check yourself
1. Is `year` quantitative or categorical? Defend your answer with a question the average would (not) answer.
2. What would go wrong if you left `month` as plain text and sorted by it?

---
## 3. DataFrame vs Series — *Am I holding a table or a single column?*

- A **DataFrame** is a 2-D table (rows × columns).
- A **Series** is a single 1-D column (with an index).

Selecting **one** column gives a Series. Selecting a **list** of columns gives a DataFrame. This distinction decides which methods you can call, so always check with `type()`.

In [19]:
# One column name in [ ]  -> Series
amt = coffee["amount"]
print(type(amt))
amt.head()

<class 'pandas.core.series.Series'>


0   -20.10
1    -7.43
2    -2.75
3    -2.75
4    -9.21
Name: amount, dtype: float64

In [20]:
# A LIST of column names in [[ ]]  -> DataFrame (even if the list has one name!)
amt_df = coffee[["amount"]]
print(type(amt_df))
amt_df.head()

<class 'pandas.core.frame.DataFrame'>


,amount
0,-20.10
1,-7.43
2,-2.75
3,-2.75
4,-9.21


In [21]:
# Q: What are the parts of a Series?
print("values:", amt.values[:5])
print("index :", amt.index[:5])
print("name  :", amt.name)
print("dtype :", amt.dtype)

values: [-20.1   -7.43  -2.75  -2.75  -9.21]
index : RangeIndex(start=0, stop=5, step=1)
name  : amount
dtype : float64


### ✅ Check yourself
What does `type(coffee[["account", "amount"]])` return? Predict, then run it.

In [22]:
type(coffee[["account", "amount"]])

<class 'pandas.core.frame.DataFrame'>

---
## 4. Accessing data with `[ ]` — *How do I grab a column, some rows, or a piece of a column?*

Square brackets do different things depending on what you put inside:

| You write | You get |
|---|---|
| `df["col"]` | one column (Series) |
| `df[["a","b"]]` | several columns (DataFrame) |
| `df[0:5]` | rows 0–4 by *position* (slice, like a list) |
| `df[condition]` | rows where the condition is `True` |
| `df["col"][0:5]` | column first, then rows |

In [23]:
# Q: What accounts were used? (one column)
coffee["account"].head()

0    Kelley
1      Eman
2      Eman
3      Eman
4    Kelley
Name: account, dtype: category
Categories (5, object): ['Business', 'Credit Card', 'Eman', 'Joint', 'Kelley']

In [24]:
# Q: What was spent and when? (several columns)
coffee[["date", "amount"]].head()

,date,amount
0,2018-10-22,-20.10
1,2018-10-29,-7.43
2,2019-06-03,-2.75
3,2019-06-10,-2.75
4,2019-08-28,-9.21


In [25]:
# Q: What are the first 5 purchases?  (a slice in [ ] selects ROWS, not columns)
coffee[0:5]

,date,description,amount,account,month,year,day_of_week
0,2018-10-22,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday
1,2018-10-29,SQ *LUCY'S COFFEE C 10/27 PURCHASE SAN LUIS OB...,-7.43,Eman,October,2018,Monday
2,2019-06-03,SQ *LUCY'S COFFEE CO 06/02 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday
3,2019-06-10,SQ *LUCY'S COFFEE CO 06/08 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday
4,2019-08-28,SQ *SCOUT COFFEE 08/27 PURCHASE SAN LUIS OBIS CA,-9.21,Kelley,August,2019,Wednesday


In [26]:
# Slice permutations — same rules as Python lists
print(coffee[:3].shape)      # first 3 rows
print(coffee[818:].shape)    # from row 818 to the end
print(coffee[-3:].shape)     # last 3 rows
print(coffee[::100].shape)   # every 100th row

(3, 7)
(3, 7)
(3, 7)
(9, 7)


In [27]:
# Q: What were the first 5 amounts?  (column, then row slice)
coffee["amount"][0:5]

0   -20.10
1    -7.43
2    -2.75
3    -2.75
4    -9.21
Name: amount, dtype: float64

In [28]:
# Q: Which purchases were over $10?  (a boolean condition selects rows)
coffee[coffee["amount"] < -10].head()

,date,description,amount,account,month,year,day_of_week
0,2018-10-22,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday
6,2020-01-10,SQ *SCOUT COFFEE 01/09 PURCHASE San Luis Obis CA,-12.91,Eman,January,2020,Friday
49,2020-09-02,SQ *SCOUT COFFEE 09/01 PURCHASE San Luis Obis CA,-10.50,Eman,September,2020,Wednesday
64,2020-11-16,SQ *KRAKEN AVILA 11/13 PURCHASE AVILA BEACH CA,-11.61,Eman,November,2020,Monday
65,2021-01-25,SQ *SCOUT COFFEE 01/24 PURCHASE San Luis Obis CA,-24.20,Eman,January,2021,Monday


In [29]:
# Q: Which purchases were made on the Joint account on a Friday?  (& = and, | = or; parentheses required)
coffee[(coffee["account"] == "Joint") & (coffee["day_of_week"] == "Friday")]

,date,description,amount,account,month,year,day_of_week


### ✅ Check yourself
1. Write one line that returns the `description` and `amount` of every purchase in `2025`.
2. What is the difference between `coffee["amount"]` and `coffee[["amount"]]`?

In [30]:
# your answer to #1 here
coffee.loc[coffee["year"] == 2025, ["description", "amount"]]

,description,amount
702,THE KBRG COFFEE LAB 01/03 MOBILE PURCHASE SAN ...,-6.50
703,SQ *SCOUT COFFEE 01/04 MOBILE PURCHASE San Lui...,-55.89
704,SQ *SCOUT COFFEE 01/05 MOBILE PURCHASE San Lui...,-3.25
705,THE KBRG COFFEE LAB 02/01 PURCHASE SAN LUIS OB...,-5.00
706,KRAKEN COFFEE COMPANY 02/02 MOBILE PURCHASE AV...,-5.25
...,...,...
816,SQ *SCOUT COFFEE 08/03 MOBILE PURCHASE San Lui...,-2.00
817,SQ *SCOUT COFFEE 08/10 MOBILE PURCHASE San Lui...,-9.10
818,SQ *SCOUT COFFEE 08/11 MOBILE PURCHASE San Lui...,-55.00
819,SQ *SCOUT COFFEE San Luis Obis CA,-4.95


---
## 5. `.loc` vs `.iloc` — *Do I want rows/columns by LABEL or by POSITION?*

`[ ]` is convenient but ambiguous (is that number a row or a column?). `.loc` and `.iloc` are explicit, and both take **`[rows, columns]`**.

| | selects by | example |
|---|---|---|
| `.loc` | **label** (index name, column name) | `df.loc[0:4, "amount"]` — **inclusive** end |
| `.iloc` | **integer position** | `df.iloc[0:5, 2]` — **exclusive** end |

Memory trick: **i**loc = **i**nteger.

In [31]:
# Q: What is the amount of the very first purchase?
print(coffee.loc[0, "amount"])    # row label 0, column label "amount"
print(coffee.iloc[0, 2])          # row position 0, column position 2

-20.1
-20.1


In [32]:
# Q: What are rows 0-4 for date and amount?
coffee.loc[0:4, ["date", "amount"]]     # .loc slice INCLUDES 4

,date,amount
0,2018-10-22,-20.10
1,2018-10-29,-7.43
2,2019-06-03,-2.75
3,2019-06-10,-2.75
4,2019-08-28,-9.21


In [33]:
coffee.iloc[0:4, [0, 2]]               # .iloc slice EXCLUDES 4  -> only 4 rows

,date,amount
0,2018-10-22,-20.10
1,2018-10-29,-7.43
2,2019-06-03,-2.75
3,2019-06-10,-2.75


### All the slice permutations with `:`
`:` alone means "everything".

In [34]:
coffee.loc[:, "amount"].head()          # all rows, one column  (same as coffee["amount"])

0   -20.10
1    -7.43
2    -2.75
3    -2.75
4    -9.21
Name: amount, dtype: float64

In [35]:
coffee.loc[5, :]                        # one row, all columns -> a Series (the row!)

date                                        2019-10-07 00:00:00
description    SQ *VERVE COFFEE RO 10/05 PURCHASE SANTA CRUZ CA
amount                                                    -5.33
account                                                    Eman
month                                                   October
year                                                       2019
day_of_week                                              Monday
Name: 5, dtype: object

In [36]:
coffee.loc[:, "amount":"month"].head()  # a RANGE of columns by label (only .loc can do this)

,amount,account,month
0,-20.10,Kelley,October
1,-7.43,Eman,October
2,-2.75,Eman,June
3,-2.75,Eman,June
4,-9.21,Kelley,August


In [37]:
coffee.iloc[-5:, :3]                    # last 5 rows, first 3 columns

,date,description,amount
816,2025-08-04,SQ *SCOUT COFFEE 08/03 MOBILE PURCHASE San Lui...,-2.00
817,2025-08-11,SQ *SCOUT COFFEE 08/10 MOBILE PURCHASE San Lui...,-9.10
818,2025-08-12,SQ *SCOUT COFFEE 08/11 MOBILE PURCHASE San Lui...,-55.00
819,2025-08-05,SQ *SCOUT COFFEE San Luis Obis CA,-4.95
820,2025-08-05,SQ *SCOUT COFFEE San Luis Obis CA,-6.50


In [38]:
coffee.iloc[::200, ::2]                 # every 200th row, every 2nd column

,date,amount,month,day_of_week
0,2018-10-22,-20.1,October,Monday
200,2021-09-07,-4.0,September,Tuesday
400,2022-09-15,-4.0,September,Thursday
600,2023-12-14,-7.2,December,Thursday
800,2025-07-08,-2.5,July,Tuesday


In [39]:
# .loc also accepts a boolean condition for rows, plus a column choice
# Q: How much was spent on Business-account purchases?
coffee.loc[coffee["account"] == "Business", ["date", "amount"]]

,date,amount
284,2022-04-05,-8.35
294,2022-05-12,-3.25
350,2022-08-11,-3.00
351,2022-08-11,-7.70
361,2022-08-17,-2.50
363,2022-08-17,-4.75
430,2022-10-05,-2.50
431,2022-10-07,-7.75
443,2022-10-18,-3.00
444,2022-10-19,-8.50


### ⚠️ Why the difference matters
After filtering or sorting, row **labels** no longer match row **positions**.

In [40]:
big = coffee[coffee["amount"] < -10]
big.head(3)

,date,description,amount,account,month,year,day_of_week
0,2018-10-22,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday
6,2020-01-10,SQ *SCOUT COFFEE 01/09 PURCHASE San Luis Obis CA,-12.91,Eman,January,2020,Friday
49,2020-09-02,SQ *SCOUT COFFEE 09/01 PURCHASE San Luis Obis CA,-10.50,Eman,September,2020,Wednesday


In [41]:
print(big.iloc[0])          # first row by POSITION -> works
print("-----")
try:
    print(big.loc[0])       # row LABELED 0 -> may not exist in the filtered frame
except KeyError as e:
    print("KeyError: no row labeled", e)

date                                         2018-10-22 00:00:00
description    SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...
amount                                                     -20.1
account                                                   Kelley
month                                                    October
year                                                        2018
day_of_week                                               Monday
Name: 0, dtype: object
-----
date                                         2018-10-22 00:00:00
description    SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...
amount                                                     -20.1
account                                                   Kelley
month                                                    October
year                                                        2018
day_of_week                                               Monday
Name: 0, dtype: object


### ✅ Check yourself
1. Using `.loc`, get `account` and `amount` for rows labeled 10 through 15.
2. Using `.iloc`, get the same cells.
3. Why does one give 6 rows and the other need `10:16`?

In [42]:
# your answers here
coffee.loc[10:15, ["account", "amount"]]     # .loc end is INCLUSIVE -> 6 rows
coffee.iloc[10:16, [3, 2]]                    # .iloc end is EXCLUSIVE -> need 10:16

,account,amount
10,Eman,-2.75
11,Eman,-3.25
12,Eman,-2.00
13,Eman,-5.75
14,Eman,-3.25
15,Eman,-3.25


---
## 6. Changing types — *How do I tell pandas what a column really is?*

`.astype()` converts a column. Three common needs:

1. **Number → string** (`astype(str)`) — e.g. a zip code or year you want treated as a label.
2. **Change the levels of a categorical** — rename, reorder, or drop levels.
3. **Quantitative → categorical** — bin a number into groups (`pd.cut`).

In [43]:
# 1) Q: How do I stop pandas from doing math on `year`?  Turn it into text (or a category).
coffee["year_str"] = coffee["year"].astype(str)
coffee[["year", "year_str"]].dtypes

year         int64
year_str    object
dtype: object

In [44]:
# Now "2022" + "!" is string work, not arithmetic
coffee["year_str"].head(3) + "!"

0    2018!
1    2018!
2    2019!
Name: year_str, dtype: object

In [45]:
# and the reverse: text -> number
coffee["year_str"].astype(int).head(3)

0    2018
1    2018
2    2019
Name: year_str, dtype: int64

In [46]:
# 2) Q: What are the levels of a categorical variable?
coffee["account"].cat.categories

Index(['Business', 'Credit Card', 'Eman', 'Joint', 'Kelley'], dtype='object')

In [47]:
# Q: How do I rename levels?  (e.g. clearer labels for a report)
coffee["account"] = coffee["account"].cat.rename_categories({"Eman": "Personal-Eman", "Kelley": "Personal-Kelley"})
coffee["account"].value_counts()

account
Personal-Eman      698
Credit Card         82
Joint               24
Business            15
Personal-Kelley      2
Name: count, dtype: int64

In [48]:
# Q: How do I reorder levels?  (put the most-used first)
coffee["account"] = coffee["account"].cat.reorder_categories(
    ["Personal-Eman", "Credit Card", "Joint", "Business", "Personal-Kelley"], ordered=True)
coffee["account"].cat.categories

Index(['Personal-Eman', 'Credit Card', 'Joint', 'Business', 'Personal-Kelley'], dtype='object')

In [49]:
# Q: How do I collapse / replace levels?  Map several levels to one new label.
coffee["account_group"] = coffee["account"].astype(str).replace(
    {"Personal-Eman": "Personal", "Personal-Kelley": "Personal"}).astype("category")
coffee["account_group"].value_counts()

account_group
Personal       700
Credit Card     82
Joint           24
Business        15
Name: count, dtype: int64

In [50]:
# 3) Q: How do I turn a quantitative variable into groups?  pd.cut with bin edges + labels
coffee["spend"] = coffee["amount"].abs()          # make dollars positive first

coffee["size"] = pd.cut(coffee["spend"],
                        bins=[0, 5, 10, 100],
                        labels=["small", "medium", "large"])
coffee[["spend", "size"]].head(10)

,spend,size
0,20.10,large
1,7.43,medium
2,2.75,small
3,2.75,small
4,9.21,medium
5,5.33,medium
6,12.91,large
7,2.75,small
8,2.75,small
9,3.25,small


In [51]:
coffee["size"].value_counts()

size
small     458
medium    280
large      83
Name: count, dtype: int64

In [52]:
# Q: Can I make groups with equal numbers of purchases instead of equal widths?  pd.qcut
coffee["spend_quartile"] = pd.qcut(coffee["spend"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
coffee["spend_quartile"].value_counts()

spend_quartile
Q1    223
Q4    205
Q3    198
Q2    195
Name: count, dtype: int64

In [53]:
# The DataFrame now states every variable's type
coffee.dtypes

date              datetime64[ns]
description               object
amount                   float64
account                 category
month                   category
year                       int64
day_of_week             category
year_str                  object
account_group           category
spend                    float64
size                    category
spend_quartile          category
dtype: object

### ✅ Check yourself
1. Bin `year` into `"2018-2020"`, `"2021-2023"`, `"2024-2025"` with `pd.cut`.
2. Why convert `year` with `astype(str)` instead of leaving it `int64`?

In [54]:
# your answer here
coffee["era"] = pd.cut(coffee["year"],
                       bins=[2017, 2020, 2023, 2025],
                       labels=["2018-2020", "2021-2023", "2024-2025"])
coffee["era"].value_counts()
# astype(str) stops pandas from treating year as a number (no meaningless averages / sums)

era
2021-2023    536
2024-2025    220
2018-2020     65
Name: count, dtype: int64

---
## 7. Creating a DataFrame from scratch — *How do I build a table without a file?*

Sometimes the data is in your head (or an assignment). The most common recipe is a **dictionary of lists**: keys are column names, lists are the column values.

In [55]:
# Q: What does it cost to make a coffee at home?
home = pd.DataFrame({
    "item":     ["beans", "milk", "filter", "sugar"],
    "cost":     [0.85, 0.30, 0.05, 0.02],
    "category": ["ingredient", "ingredient", "supply", "ingredient"],
})
home

,item,cost,category
0,beans,0.85,ingredient
1,milk,0.30,ingredient
2,filter,0.05,supply
3,sugar,0.02,ingredient


In [56]:
# Everything from above works on it immediately
print(home.shape)
print(home.dtypes)
home["category"] = home["category"].astype("category")
home.loc[home["category"] == "ingredient", "cost"].sum()

(4, 3)
item         object
cost        float64
category     object
dtype: object


np.float64(1.17)

In [57]:
# Another recipe: a list of dictionaries (one dict per row)
pd.DataFrame([
    {"item": "beans", "cost": 0.85},
    {"item": "milk",  "cost": 0.30},
])

,item,cost
0,beans,0.85
1,milk,0.30


In [58]:
# A Series on its own, and then turning it into a DataFrame
s = pd.Series([3, 5, 2], index=["Mon", "Tue", "Wed"], name="coffees")
print(type(s))
s.to_frame()

<class 'pandas.core.series.Series'>


,coffees
Mon,3
Tue,5
Wed,2


In [59]:
# Adding a column to an existing DataFrame is just assignment
home["cost_per_week"] = home["cost"] * 7
home

,item,cost,category,cost_per_week
0,beans,0.85,ingredient,5.95
1,milk,0.30,ingredient,2.10
2,filter,0.05,supply,0.35
3,sugar,0.02,ingredient,0.14


### ✅ Check yourself
Build a DataFrame with 4 rows about coffee shops: `name` (text), `rating` (quantitative), `has_wifi` (categorical: yes/no). Declare `has_wifi` as a category, then print `.dtypes` and `.shape`.

In [60]:
# your DataFrame here
shops = pd.DataFrame({
    "name":     ["Scout", "Linnaea's", "Kreuzberg", "Black Horse"],
    "rating":   [4.6, 4.4, 4.5, 4.2],
    "has_wifi": ["yes", "yes", "yes", "no"],
})
shops["has_wifi"] = shops["has_wifi"].astype("category")
print(shops.dtypes)
print(shops.shape)

name          object
rating       float64
has_wifi    category
dtype: object
(4, 3)


---
## Summary — the questions and their answers

| Question | Tool |
|---|---|
| How do I load a CSV? | `pd.read_csv()` |
| How big is it? What's in it? | `.shape`, `type()`, `.dtypes`, `.info()`, `.head()` |
| What kind of variable is this, really? | quantitative / categorical / date-time → `astype("category")`, `pd.to_datetime()` |
| Table or column? | DataFrame vs Series — `df["col"]` vs `df[["col"]]` |
| How do I grab columns / rows / filtered rows? | `[ ]` with a name, a list, a slice, or a condition |
| By label or by position? | `.loc[rows, cols]` vs `.iloc[rows, cols]` |
| How do I change a column's type? | `.astype(str)`, `.cat.rename_categories`, `pd.cut` / `pd.qcut` |
| How do I build a table from scratch? | `pd.DataFrame({...})` |